In [1]:
from pytrends.request import TrendReq
import pandas as pd
import time
import random
from datetime import datetime, timedelta
import os

# -----------------------------
# CONFIG
# -----------------------------

keywords = [
    'gold price',
    'gold forecast',
    'buy gold',
    'inflation',
    'safe haven'
]

start_date = datetime(2015, 1, 1)
end_date = datetime(2024, 12, 31)
window_size = 180  # ~6 months

OUTPUT_FILE = 'google_trends_gold_keywords_daily_2015_2024.csv'

BASE_SLEEP = 10
RATE_LIMIT_SLEEP_MIN = 120
RATE_LIMIT_SLEEP_MAX = 300

# -----------------------------
# INIT PYTRENDS
# -----------------------------

pytrend = TrendReq(
    hl='en-US',
    tz=360,
    timeout=(10, 25)
)

# -----------------------------
# LOAD EXISTING DATA (if any)
# -----------------------------

if os.path.exists(OUTPUT_FILE):
    final_df = pd.read_csv(OUTPUT_FILE, parse_dates=['date'])
    completed_keywords = set(final_df.columns) - {'date'}
    print(f"Resuming. Found existing keywords: {completed_keywords}")
else:
    final_df = None
    completed_keywords = set()
    print("Starting fresh run.")

# -----------------------------
# MAIN LOOP
# -----------------------------

for kw in keywords:

    if kw in completed_keywords:
        print(f"Skipping already completed keyword: {kw}")
        continue

    print(f"\nProcessing keyword: {kw}")
    all_dfs = []
    current_start = start_date

    while current_start < end_date:
        current_end = min(current_start + timedelta(days=window_size), end_date)
        timeframe = f"{current_start.strftime('%Y-%m-%d')} {current_end.strftime('%Y-%m-%d')}"

        print(f"  Fetching {timeframe}")

        try:
            pytrend.build_payload(
                kw_list=[kw],
                timeframe=timeframe,
                geo=''
            )

            df = pytrend.interest_over_time()

            if not df.empty:
                df = df[[kw]]
                all_dfs.append(df)

            time.sleep(BASE_SLEEP)

        except Exception as e:
            if "429" in str(e):
                wait = random.randint(RATE_LIMIT_SLEEP_MIN, RATE_LIMIT_SLEEP_MAX)
                print(f"⛔ Rate limited. Sleeping for {wait} seconds...")
                time.sleep(wait)
                continue
            else:
                print(f"⚠️ Error: {e}. Retrying in 10s...")
                time.sleep(10)
                continue

        current_start = current_end + timedelta(days=1)

    # -----------------------------
    # MERGE & SAVE CHECKPOINT
    # -----------------------------

    if not all_dfs:
        print(f"No data collected for keyword: {kw}")
        continue

    kw_df = pd.concat(all_dfs)
    kw_df = kw_df[~kw_df.index.duplicated(keep='first')]
    kw_df.reset_index(inplace=True)

    if final_df is None:
        final_df = kw_df
    else:
        final_df = final_df.merge(kw_df, on='date', how='outer')

    final_df.sort_values('date', inplace=True)

    # SAVE AFTER EACH KEYWORD
    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"✅ Saved progress after keyword: {kw}")

# -----------------------------
# DONE
# -----------------------------

print("\nAll keywords processed.")
print(f"Final dataset saved to: {OUTPUT_FILE}")
print(final_df.head())


Starting fresh run.

Processing keyword: gold price
  Fetching 2015-01-01 2015-06-30
  Fetching 2015-07-01 2015-12-28
  Fetching 2015-12-29 2016-06-26
  Fetching 2016-06-27 2016-12-24
  Fetching 2016-12-25 2017-06-23
  Fetching 2017-06-24 2017-12-21
  Fetching 2017-12-22 2018-06-20
  Fetching 2018-06-21 2018-12-18
  Fetching 2018-12-19 2019-06-17
  Fetching 2019-06-18 2019-12-15
  Fetching 2019-12-16 2020-06-13
  Fetching 2020-06-14 2020-12-11
  Fetching 2020-12-12 2021-06-10
  Fetching 2021-06-11 2021-12-08
  Fetching 2021-12-09 2022-06-07
  Fetching 2022-06-08 2022-12-05
  Fetching 2022-12-06 2023-06-04
  Fetching 2023-06-05 2023-12-02
  Fetching 2023-12-03 2024-05-31
  Fetching 2024-06-01 2024-11-28
  Fetching 2024-11-29 2024-12-31
✅ Saved progress after keyword: gold price

Processing keyword: gold forecast
  Fetching 2015-01-01 2015-06-30
  Fetching 2015-07-01 2015-12-28
  Fetching 2015-12-29 2016-06-26
  Fetching 2016-06-27 2016-12-24
  Fetching 2016-12-25 2017-06-23
  Fetching 2